In [ ]:
import math
import os
import sys
from copy import deepcopy
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.ndimage import affine_transform
from Bio.PDB import PDBParser


def collect_atoms(model):
    return list(model.get_atoms())


def collect_residues(chain):
    return list(chain.get_residues())


def get_plot_data(residues_dict):
    interface_l, interface_r, complex_points = [], [], []

    for designation in ("l", "r"):
        bound_residues = residues_dict[f"{designation}_b"]
        unbound_residues = residues_dict[f"{designation}_u"]

        for bound in (True, False):
            residues = bound_residues if bound else unbound_residues
            for res in residues.values():
                x, y, z = res["c_a_coords"]
                if bound:
                    color_val = "ligand" if designation == "l" else "receptor"
                    if res["is_interface"]:
                        color_val = f"{color_val}_interface"
                    complex_points.append(
                        {
                            "x_coord": float(x),
                            "y_coord": float(y),
                            "z_coord": float(z),
                            "color_val": color_val,
                        }
                    )
                if not res["is_interface"]:
                    continue
                color_val = res["res_num"] % 5
                point_data = {
                    "x_coord": float(x),
                    "y_coord": float(y),
                    "z_coord": float(z),
                    "color_val": int(color_val),
                }
                if designation == "l":
                    interface_l.append(point_data)
                else:
                    interface_r.append(point_data)

    return interface_l, interface_r, complex_points


def plot_3d(plot_dict: List[Dict], output_fname: str, figures_path: str):
    if not plot_dict:
        return
    df = pd.DataFrame(plot_dict)
    fig = px.scatter_3d(
        df,
        x="x_coord",
        y="y_coord",
        z="z_coord",
        color="color_val",
        labels={"color_val": "value"},
    )
    filename = os.path.join(figures_path, f"{output_fname}.html")
    pio.write_html(fig, filename, auto_open=False)


def apply_transformation(residues_dict: Dict, rotation: np.ndarray, translation: np.ndarray):
    transformed = {}
    midpt = calc_midpoint([val["c_a_coords"] for val in residues_dict.values()])
    rotation = np.asarray(rotation, dtype=float)
    translation = np.asarray(translation, dtype=float)
    for key, val in residues_dict.items():
        coords = np.asarray(val["c_a_coords"], dtype=float)
        new_coords = coords - midpt
        new_coords = rotation @ new_coords
        new_coords = new_coords + translation + midpt
        transformed[key] = {
            "is_interface": bool(val["is_interface"]),
            "c_a_coords": new_coords,
            "res_num": val["res_num"],
            "coords": val["coords"],
        }
    return transformed


def get_rotation(B):
    B = np.asarray(B, dtype=float)
    if np.linalg.norm(B) == 0:
        raise ValueError("Rotation vector must be non-zero.")
    A = np.array([1.0, 0.0, 0.0])
    dot_AB = float(np.dot(A, B))
    cross_AB = np.cross(A, B)
    norm_cross = np.linalg.norm(cross_AB)

    if np.isclose(norm_cross, 0.0):
        return np.eye(3)

    secondary = B - dot_AB * A
    secondary_norm = np.linalg.norm(secondary)
    if np.isclose(secondary_norm, 0.0):
        return np.eye(3)

    secondary /= secondary_norm
    Fi = np.column_stack([A, secondary, np.cross(B, A)])
    G = np.array(
        [[dot_AB, -norm_cross, 0.0], [norm_cross, dot_AB, 0.0], [0.0, 0.0, 1.0]]
    )
    return Fi @ G @ np.linalg.inv(Fi)


def calc_midpoint(coords):
    coords_array = np.vstack(coords)
    mins = coords_array.min(axis=0)
    maxs = coords_array.max(axis=0)
    return 0.5 * (mins + maxs)


def calc_interface_rmsd(proteins_dict: Dict):
    total = 0.0
    count = 0

    for designation in ("l", "r"):
        bound_residues = proteins_dict[f"{designation}_b"]
        unbound_residues = proteins_dict[f"{designation}_u"]

        for res_key, res_bound in bound_residues.items():
            if not res_bound["is_interface"]:
                continue
            res_unbound = unbound_residues.get(res_key)
            if res_unbound is None:
                continue
            diff = res_bound["c_a_coords"] - res_unbound["c_a_coords"]
            dist2 = np.linalg.norm(diff) ** 2
            total += dist2
            count += 1

    if count == 0:
        return float("nan")
    return math.sqrt(total / count)


def calc_dist_matrix(atoms):
    n = len(atoms)
    dist_matrix = np.zeros((n, n), dtype=float)
    for k in range(n):
        for j in range(k):
            dist = np.linalg.norm(atoms[j].get_coord() - atoms[k].get_coord())
            dist_matrix[j, k] = dist
            dist_matrix[k, j] = dist
    return dist_matrix


def get_sphere_points(n: int):
    dl = math.pi * (3.0 - math.sqrt(5.0))
    dz = 2.0 / n
    longitude = 0.0
    z = 1.0 - dz / 2.0
    coords = np.zeros((n, 3), dtype=float)

    for k in range(n):
        r = math.sqrt(1.0 - z * z)
        coords[k, 0] = math.cos(longitude) * r
        coords[k, 1] = math.sin(longitude) * r
        coords[k, 2] = z
        z -= dz
        longitude += dl

    return coords


def get_asa(
    atoms_coords: List[np.ndarray],
    radius_list: List[float],
    dist_matrix: np.ndarray,
    sphere_points=None,
):
    if sphere_points is None:
        sphere_points = get_sphere_points(100)
    sphere_points = np.asarray(sphere_points, dtype=float)
    probe_radius = 1.4
    max_radius = 2.0 * (max(radius_list) + probe_radius)
    neighbors = [
        np.where(dist_matrix[i, :] <= max_radius + 0.01)[0]
        for i in range(dist_matrix.shape[0])
    ]
    atoms_coords_array = np.asarray(atoms_coords, dtype=float)
    n_points = sphere_points.shape[0]
    atom_asa = np.zeros(len(atoms_coords), dtype=float)

    for atom_idx in range(len(atoms_coords)):
        atom_radius = radius_list[atom_idx]
        neighbor_idxs = neighbors[atom_idx]
        atom_sphere_points = (
            atoms_coords_array[atom_idx]
            + (atom_radius + probe_radius) * sphere_points
        )
        points_inaccessible = np.zeros(n_points, dtype=bool)

        for neighbor_idx in neighbor_idxs:
            if neighbor_idx == atom_idx:
                continue
            neighbor_atom = atoms_coords_array[neighbor_idx]
            neighbor_radius = radius_list[neighbor_idx]
            distances = np.linalg.norm(atom_sphere_points - neighbor_atom, axis=1)
            blocked = distances < (neighbor_radius + probe_radius)
            points_inaccessible |= blocked

        percent_accessible = 1.0 - (np.sum(points_inaccessible) / n_points)
        surface_area = 4 * math.pi * (atom_radius + probe_radius) ** 2
        atom_asa[atom_idx] = percent_accessible * surface_area

    return atom_asa


def rotate_volume(volume: np.ndarray, rotation_matrix: np.ndarray):
    matrix = np.asarray(rotation_matrix, dtype=float).T
    center = (np.array(volume.shape, dtype=float) - 1.0) / 2.0
    offset = center - matrix @ center
    return affine_transform(
        volume,
        matrix,
        offset=offset,
        order=0,
        mode="constant",
        cval=0.0,
        prefilter=True,
    )


def scan_6d(rotation_vectors, receptor_volume, ligand_volume):
    top_scores = []
    vol_shape = receptor_volume.shape
    n_voxels = np.prod(vol_shape)
    dft_lc = np.fft.fftn(receptor_volume)
    n_vectors = rotation_vectors.shape[0]

    for i in range(n_vectors):
        if (i + 1) % 100 == 0:
            print(
                f"Calculating scores (rotation {i+1} out of {n_vectors})"
            )
        v = rotation_vectors[i]
        rotation = get_rotation(v)
        ligand_rotated = rotate_volume(ligand_volume, rotation)
        ift_rc = n_voxels * np.fft.ifftn(ligand_rotated)
        inv_val = np.fft.ifftn(ift_rc * dft_lc)
        inv_val = np.fft.fftshift(inv_val)
        ssc = np.real(inv_val) - np.imag(inv_val)
        cutoff = np.percentile(ssc, 99.9)
        top_idxs = np.argwhere(ssc > cutoff)
        for idx in top_idxs:
            idx_tuple = tuple(int(x) for x in idx)
            top_scores.append((i, idx_tuple, float(ssc[idx_tuple])))

    return top_scores


def get_element_symbol(atom):
    element = atom.element.strip().upper() if atom.element else ""
    if not element:
        name = atom.get_name().strip().upper()
    else:
        name = element
    return name[:2]

import time


def main():
    if len(sys.argv) < 2:
        raise ValueError("Please provide a protein complex PDB identifier as an argument.")

    protein_complex_pdb_id = sys.argv[1]
    figures_path = f"{protein_complex_pdb_id}_figures"
    os.makedirs(figures_path, exist_ok=True)

    print("Loading data...")
    parser = PDBParser(QUIET=True)
    proteins_dict = {}

    for designation in ("l_u", "r_u", "l_b", "r_b"):
        pdb_path = os.path.join(
            "protein_data", f"{protein_complex_pdb_id}_{designation}.pdb"
        )
        structure = parser.get_structure(
            f"{protein_complex_pdb_id}_{designation}", pdb_path
        )
        models = list(structure.get_models())
        if len(models) != 1:
            raise ValueError(
                f"PDB file {pdb_path} should contain exactly one model."
            )
        proteins_dict[designation] = models[0]

    ligand_unbound_chains = {chain.id for chain in proteins_dict["l_u"]}
    ligand_bound_chains = {chain.id for chain in proteins_dict["l_b"]}
    if ligand_unbound_chains != ligand_bound_chains:
        raise ValueError(
            "Bound and unbound ligand proteins have different chain IDs."
        )

    receptor_unbound_chains = {chain.id for chain in proteins_dict["r_u"]}
    receptor_bound_chains = {chain.id for chain in proteins_dict["r_b"]}
    if receptor_unbound_chains != receptor_bound_chains:
        raise ValueError(
            "Bound and unbound receptor proteins have different chain IDs."
        )

    ligand_atoms = collect_atoms(proteins_dict["l_u"])
    receptor_atoms = collect_atoms(proteins_dict["r_u"])
    ligand_coords = [atom.get_coord() for atom in ligand_atoms]
    receptor_coords = [atom.get_coord() for atom in receptor_atoms]

    ligand_midpt = calc_midpoint(ligand_coords)
    receptor_midpt = calc_midpoint(receptor_coords)
    ligand_coords = [coord - ligand_midpt for coord in ligand_coords]
    receptor_coords = [coord - receptor_midpt for coord in receptor_coords]

    all_residues_dict = {}
    for key, model in proteins_dict.items():
        residues = {}
        for chain in model:
            for residue in chain:
                if residue.id[0] != " ":
                    continue
                if "CA" not in residue:
                    continue
                chain_id = chain.id
                res_num = residue.id[1]
                coords = [
                    atom.get_coord() - receptor_midpt for atom in residue.get_atoms()
                ]
                residues[(chain_id, res_num)] = {
                    "coords": coords,
                    "res_num": res_num,
                    "is_interface": False,
                    "c_a_coords": residue["CA"].get_coord() - receptor_midpt,
                }
        all_residues_dict[key] = residues

    print("----------------------------------------------------------------------------")
    print("Extracting interface and calculating I-RMSD...")

    interface_max_dist = 10.0
    for res_key_l, res_l in all_residues_dict["l_b"].items():
        coords_l = res_l["coords"]
        for res_key_r, res_r in all_residues_dict["r_b"].items():
            coords_r = res_r["coords"]
            is_interface = any(
                np.linalg.norm(coord2 - coord1) <= interface_max_dist
                for coord1 in coords_l
                for coord2 in coords_r
            )
            if not is_interface:
                continue

            all_residues_dict["r_b"][res_key_r]["is_interface"] = True
            if res_key_r in all_residues_dict["r_u"]:
                all_residues_dict["r_u"][res_key_r]["is_interface"] = True

            all_residues_dict["l_b"][res_key_l]["is_interface"] = True
            if res_key_l in all_residues_dict["l_u"]:
                all_residues_dict["l_u"][res_key_l]["is_interface"] = True

    interface_l, interface_r, complex_points = get_plot_data(all_residues_dict)
    plot_3d(
        interface_l,
        f"ligand_compare_{protein_complex_pdb_id}",
        figures_path,
    )
    plot_3d(
        interface_r,
        f"receptor_compare_{protein_complex_pdb_id}",
        figures_path,
    )
    plot_3d(
        complex_points,
        f"complex_{protein_complex_pdb_id}",
        figures_path,
    )

    irmsd = calc_interface_rmsd(all_residues_dict)
    print(
        f"The I-RMSD for {protein_complex_pdb_id} is {irmsd}"
    )
    print("----------------------------------------------------------------------------")
    print("Discretizing proteins into 3D volumes...")

    init_rotation_vec = np.array([0.667, 0.606, 0.432], dtype=float)
    init_rotation_vec /= np.linalg.norm(init_rotation_vec)
    inv_rotation_vec = np.array(
        [init_rotation_vec[0], -init_rotation_vec[1], -init_rotation_vec[2]],
        dtype=float,
    )
    init_rotation = get_rotation(init_rotation_vec)
    print(
        f"Ligand rotation vector: {np.round(inv_rotation_vec, 3).tolist()}"
    )

    translation = receptor_midpt - ligand_midpt
    all_residues_dict["l_u"] = apply_transformation(
        all_residues_dict["l_u"], init_rotation, translation
    )
    ligand_coords = [init_rotation @ coord for coord in ligand_coords]

    atom_radius_dict = {
        "C": 1.700,
        "N": 1.550,
        "O": 1.520,
        "S": 1.800,
        "NA": 2.270,
        "MG": 1.730,
    }

    def radius_for_atom(atom):
        element = get_element_symbol(atom)
        if element not in atom_radius_dict:
            raise KeyError(f"Missing radius for atom element '{element}'")
        return atom_radius_dict[element]

    ligand_radius_list = [radius_for_atom(atom) for atom in ligand_atoms]
    receptor_radius_list = [radius_for_atom(atom) for atom in receptor_atoms]

    ligand_dist_matrix = calc_dist_matrix(ligand_atoms)
    receptor_dist_matrix = calc_dist_matrix(receptor_atoms)

    n_points = 1000
    sphere_points = get_sphere_points(n_points)

    ligand_atoms_asa = get_asa(
        ligand_coords, ligand_radius_list, ligand_dist_matrix, sphere_points
    )
    receptor_atoms_asa = get_asa(
        receptor_coords, receptor_radius_list, receptor_dist_matrix, sphere_points
    )

    ligand_max_dist = np.max(ligand_dist_matrix)
    receptor_max_dist = np.max(receptor_dist_matrix)

    volume_size = ligand_max_dist + receptor_max_dist + 2.0
    volume_dim = 128
    center_val = volume_dim // 2
    voxel_size = volume_size / (volume_dim - 1)
    vol_start = -0.5 * volume_size

    print(f"voxel size: {voxel_size}")

    receptor_volume = np.zeros((volume_dim, volume_dim, volume_dim), dtype=float)
    ligand_volume = np.zeros((volume_dim, volume_dim, volume_dim), dtype=float)
    vols = {"l": ligand_volume, "r": receptor_volume}

    asa_min = 1.0
    rr_iter1 = [
        math.sqrt(1.5) * receptor_radius_list[i]
        if receptor_atoms_asa[i] < asa_min
        else math.sqrt(0.8) * receptor_radius_list[i]
        for i in range(len(receptor_atoms_asa))
    ]
    rr_iter2 = [
        0.0
        if receptor_atoms_asa[i] < asa_min
        else receptor_radius_list[i] + 3.4
        for i in range(len(receptor_atoms_asa))
    ]
    lr_iter1 = [
        math.sqrt(1.5) * ligand_radius_list[i]
        if ligand_atoms_asa[i] < asa_min
        else 0.0
        for i in range(len(ligand_atoms_asa))
    ]
    lr_iter2 = [
        0.0 if ligand_atoms_asa[i] < asa_min else ligand_radius_list[i]
        for i in range(len(ligand_atoms_asa))
    ]

    for iter_num, designation, radius_list in [
        (1, "r", rr_iter1),
        (2, "r", rr_iter2),
        (1, "l", lr_iter1),
        (2, "l", lr_iter2),
    ]:
        atoms = receptor_coords if designation == "r" else ligand_coords
        volume = vols[designation]
        set_val = 2.0 if iter_num == 1 else 1.0
        for atom_idx, atom in enumerate(atoms):
            atom_r = radius_list[atom_idx]
            if atom_r <= 0.0:
                continue
            vol_idxs = [
                int(math.floor((atom[i] - vol_start) / voxel_size)) for i in range(3)
            ]
            n_neighbor_voxels = int(math.ceil(atom_r / voxel_size))
            start_end_idx = []
            for dim_idx in range(3):
                start = max(0, vol_idxs[dim_idx] - n_neighbor_voxels)
                end = min(volume_dim - 1, vol_idxs[dim_idx] + n_neighbor_voxels)
                start_end_idx.append((start, end))

            for x in range(start_end_idx[0][0], start_end_idx[0][1] + 1):
                for y in range(start_end_idx[1][0], start_end_idx[1][1] + 1):
                    for z in range(start_end_idx[2][0], start_end_idx[2][1] + 1):
                        if (
                            iter_num == 2
                            and designation == "r"
                            and volume[x, y, z] == 2.0
                        ):
                            continue
                        voxel_idx = np.array([x, y, z], dtype=float)
                        voxel_center = vol_start + voxel_size * (voxel_idx + 0.5)
                        center_dist = np.linalg.norm(voxel_center - atom)
                        if center_dist <= atom_r:
                            volume[x, y, z] = set_val

    change_idxs = []
    for x in range(1, volume_dim - 1):
        for y in range(1, volume_dim - 1):
            for z in range(1, volume_dim - 1):
                if ligand_volume[x, y, z] != 2.0:
                    continue
                neighbor_vals = []
                for x1 in range(x - 1, x + 2):
                    for y1 in range(y - 1, y + 2):
                        for z1 in range(z - 1, z + 2):
                            if not (x1 == x or y1 == y or z1 == z):
                                continue
                            neighbor_vals.append(ligand_volume[x1, y1, z1])
                zero_neighbors = sum(val == 0.0 for val in neighbor_vals)
                if zero_neighbors >= 2:
                    change_idxs.append((x, y, z))

    for x, y, z in change_idxs:
        ligand_volume[x, y, z] = 1.0

    slice_idx = center_val
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=["Ligand slice", "Receptor slice"],
    )
    fig.add_trace(
        go.Heatmap(z=ligand_volume[:, :, slice_idx]),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Heatmap(z=receptor_volume[:, :, slice_idx]),
        row=1,
        col=2,
    )
    fig.update_layout(width=1600, height=800)
    pio.write_html(
        fig,
        os.path.join(figures_path, f"vol_slices_{protein_complex_pdb_id}.html"),
        auto_open=False,
    )
    print("----------------------------------------------------------------------------")
    print("Scanning 6D search space...")

    rho = 9j
    ligand_volume_complex = np.where(
        ligand_volume > 1.0, rho, ligand_volume
    ).astype(np.complex128)
    receptor_volume_complex = np.where(
        receptor_volume > 1.0, rho, receptor_volume
    ).astype(np.complex128)

    n_rotations = 500
    retain_n_scores = 2000
    rotation_vectors = get_sphere_points(n_rotations)
    top_scores = scan_6d(rotation_vectors, receptor_volume_complex, ligand_volume_complex)
    top_scores = sorted(top_scores, key=lambda x: x[2], reverse=True)[:retain_n_scores]

    print("----------------------------------------------------------------------------")
    print("Evaluating candidate transformations...")

    cutoff_a = 2.5
    l_u_copy = deepcopy(all_residues_dict["l_u"])
    close_irmsds = []

    for score_rank, (rotation_idx, t, score) in enumerate(top_scores, start=1):
        translation_vector = (
            voxel_size
            * (
                np.array(t, dtype=float)
                - np.array([center_val, center_val, center_val], dtype=float)
            )
        )
        rotation_vector = rotation_vectors[rotation_idx]
        transformation = apply_transformation(
            l_u_copy, get_rotation(rotation_vector), translation_vector
        )
        all_residues_dict["l_u"] = transformation
        i_rmsd = calc_interface_rmsd(all_residues_dict)
        if i_rmsd < cutoff_a:
            close_irmsds.append((score_rank, rotation_vector.copy(), score, i_rmsd))

    close_irmsds.sort(key=lambda x: x[3], reverse=True)
    total_close = len(close_irmsds)
    print(
        f"{total_close} out of {retain_n_scores} with I-RMSD < {cutoff_a}"
    )

    for candidate_idx, (rank, rotation_vector, score, i_rmsd) in enumerate(
        close_irmsds, start=1
    ):
        dot_val = float(np.dot(rotation_vector, inv_rotation_vec))
        dot_val = np.clip(dot_val, -1.0, 1.0)
        theta = math.degrees(math.acos(dot_val))

        print(f"\n-----Candidate {candidate_idx}-----")
        print(f"Score rank: {rank}")
        print(f"Score: {score}")
        print(f"I-RMSD: {i_rmsd}")
        print(f"Rotation vector: {np.round(rotation_vector, 3).tolist()}")
        print(
            f"Angle formed with correct inverse vector (degrees): {theta}"
        )


import time

if __name__ == "__main__":
    start_time = time.time()
    main()

    end_time = time.time()

    total_seconds = end_time - start_time
    minutes = int(total_seconds // 60)
    seconds = total_seconds % 60

    print(f"\nTotal execution time: {minutes} min {seconds:.2f} sec")